# Solar Stereoscopy and Tomography: Implementation / 태양 입체법과 토모그래피: 구현

**Paper**: Aschwanden, M. J. (2011), *Living Reviews in Solar Physics* 8, 5

## Overview / 개요

**English.** This notebook implements the core geometric concepts of the paper:
1. **Epipolar geometry** — the plane family that reduces 2D correspondence search to 1D
2. **Coordinate transformations** between STEREO-A and STEREO-B viewpoints
3. **Tie-point triangulation** — the analytic triangulation formulas (Eqs. 15-23)
4. **Visualisation** of the tie-point method on a synthetic coronal loop
5. **Back-projection tomography** — the simplest tomographic inversion, demonstrated on a 2D phantom

**한국어.** 본 노트북은 논문의 핵심 기하 개념을 구현한다:
1. **에피폴라 기하** — 2D 대응점 탐색을 1D로 축소하는 평면 계열
2. **STEREO-A · STEREO-B 시점 간 좌표 변환**
3. **타이-포인트 삼각측량** — 해석적 공식 (Eqs. 15–23)
4. **합성 코로나 루프에 대한 타이-포인트 법 시각화**
5. **역투영 토모그래피** — 2D 팬텀으로 가장 단순한 토모그래피 반전 시연

## 1. Imports and Constants / 임포트와 상수

**English.** We use NumPy for linear algebra and Matplotlib for 3D/2D visualization. SciPy is used for the Radon transform in the tomography demo.

**한국어.** 선형대수는 NumPy, 3D/2D 시각화는 Matplotlib, 토모그래피 데모의 라돈 변환은 SciPy를 사용한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Physical constants in CGS / SI units.
R_SUN_KM = 696_000.0  # Solar radius in km.
AU_KM = 1.496e8  # Astronomical unit in km.
ARCSEC_PER_DEG = 3600.0

# STEREO-era canonical geometry: both spacecraft near 1 AU, separation angle variable.
D_A_KM = AU_KM  # STEREO-A Sun distance.
D_B_KM = AU_KM  # STEREO-B Sun distance.

plt.rcParams["figure.dpi"] = 110
np.set_printoptions(precision=4, suppress=True)
print(f"R_sun = {R_SUN_KM:,} km")
print(f"1 AU  = {AU_KM:,} km")


## 2. Spacecraft Geometry / 위성 기하

**English.** Place the Sun at the origin. STEREO-A lies on the negative-Z axis at distance d_A; STEREO-B lies in the XZ-plane at distance d_B, rotated by +alpha_sep degrees from A (counter-clockwise as seen from north). This matches Aschwanden (2011) Figure 9.

**한국어.** 태양을 원점에 둔다. STEREO-A는 -Z축 방향으로 d_A 거리, STEREO-B는 XZ 평면 안에서 A로부터 +alpha_sep 도 회전한 위치(북쪽에서 볼 때 반시계). Aschwanden (2011) Figure 9와 일치.

In [ ]:
def spacecraft_positions(d_a: float, d_b: float, alpha_sep_deg: float) -> tuple[np.ndarray, np.ndarray]:
    """Compute STEREO-A and STEREO-B positions in the Sun-centred heliocentric frame.

    Convention follows Aschwanden (2011) Figure 9: STEREO-A sits on the -Z axis
    (looking toward +Z at the Sun), and STEREO-B is rotated by alpha_sep degrees
    counter-clockwise around the +Y axis.

    Args:
        d_a: STEREO-A Sun distance in km.
        d_b: STEREO-B Sun distance in km.
        alpha_sep_deg: Spacecraft separation angle in degrees.

    Returns:
        (r_a, r_b): Each a (3,) np.ndarray giving (x, y, z) in km.
    """
    alpha = np.deg2rad(alpha_sep_deg)
    r_a = np.array([0.0, 0.0, -d_a])
    r_b = np.array([d_b * np.sin(alpha), 0.0, -d_b * np.cos(alpha)])
    return r_a, r_b


alpha_sep_deg = 30.0
r_a, r_b = spacecraft_positions(D_A_KM, D_B_KM, alpha_sep_deg)
print(f"STEREO-A at {r_a} km")
print(f"STEREO-B at {r_b} km")
print(f"Separation angle = {alpha_sep_deg} deg")

## 3. Tie-Point Triangulation / 타이-포인트 삼각측량

**English.** Given observed pixel angles (alpha_A, delta_A) from STEREO-A and (alpha_B, delta_B) from STEREO-B, recover the 3D source position (x, y, z) using Aschwanden (2011) Eqs. 15-23. No iteration is required — all formulas are closed-form.

**한국어.** STEREO-A의 관측 픽셀 각 (alpha_A, delta_A)와 STEREO-B의 (alpha_B, delta_B)가 주어지면, Aschwanden (2011)의 Eqs. 15–23로 3D 광원 위치 (x, y, z)를 복원. 모두 닫힌 꼴, 반복 불필요.

In [ ]:
def tie_point_triangulate(
    alpha_a: float,
    delta_a: float,
    alpha_b: float,
    alpha_sep: float,
    d_a: float = D_A_KM,
    d_b: float = D_B_KM,
) -> tuple[float, float, float, float, float]:
    """Analytic tie-point triangulation (Aschwanden 2011, Eqs. 15-23).

    Args:
        alpha_a: Angle in X-direction (X-pixel offset from Sun centre) from STEREO-A, radians.
        delta_a: Angle in Y-direction from STEREO-A, radians.
        alpha_b: Angle in X-direction from STEREO-B, radians.
        alpha_sep: Spacecraft separation angle, radians.
        d_a: STEREO-A Sun distance in km.
        d_b: STEREO-B Sun distance in km.

    Returns:
        (x, y, z, r, h): 3D source position, radial distance from Sun centre, altitude above surface, in km.
    """
    # Interior angles of the triangles O-A-x_A and O-B-x_B (Eqs. 15-16).
    gamma_a = np.pi / 2 - alpha_a
    gamma_b = np.pi / 2 - alpha_b - alpha_sep

    # Image-plane X-coordinates of the tie point (Eqs. 17-18).
    x_a = d_a * np.tan(alpha_a)
    x_b = d_b * np.sin(alpha_b) / np.sin(gamma_b)

    # Source (x, z) in the epipolar XZ plane (Eqs. 19-20).
    x = (x_b * np.tan(gamma_b) - x_a * np.tan(gamma_a)) / (np.tan(gamma_b) - np.tan(gamma_a))
    z = (x_a - x) * np.tan(gamma_a)

    # Source y from the YZ plane (Eq. 21).
    y = (d_a - z) * np.tan(delta_a)

    # Distance and altitude (Eqs. 22-23).
    r = np.sqrt(x ** 2 + y ** 2 + z ** 2)
    h = r - R_SUN_KM
    return x, y, z, r, h


### 3.1 Sanity-check with the Worked Example from notes.md / notes.md의 풀이 예제 검증

**English.** Using alpha_A = 0.25°, alpha_B = -0.15°, delta_A = 0.10°, alpha_sep = 30° and d_A = d_B = 1 AU, we expect an altitude around ~14 Mm above the solar surface (our analytic hand calculation).

**한국어.** alpha_A = 0.25°, alpha_B = -0.15°, delta_A = 0.10°, alpha_sep = 30°, d_A = d_B = 1 AU에서, notes.md의 수기 계산과 동일한 ~14 Mm 고도가 나와야 한다.

In [ ]:
# Parameters from the worked example in notes.md.
alpha_a_deg, delta_a_deg = 0.25, 0.10
alpha_b_deg = -0.15
alpha_sep_deg = 30.0

x, y, z, r, h = tie_point_triangulate(
    np.deg2rad(alpha_a_deg),
    np.deg2rad(delta_a_deg),
    np.deg2rad(alpha_b_deg),
    np.deg2rad(alpha_sep_deg),
)

print(f"Source position (x, y, z) = ({x:.3e}, {y:.3e}, {z:.3e}) km")
print(f"Distance from Sun centre  r = {r:.3e} km = {r/R_SUN_KM:.3f} R_sun")
print(f"Altitude above surface    h = {h:.3e} km = {h/1000:.1f} Mm")

## 4. Round-Trip Test: Project and Reconstruct / 왕복 검증: 투영 후 재구성

**English.** A stronger validation: generate a 3D point with known (x, y, z), project it onto both STEREO-A and STEREO-B (computing the pixel angles a spacecraft would measure), then run `tie_point_triangulate` and confirm we recover the original coordinates.

**한국어.** 더 강한 검증: 알려진 (x, y, z)의 3D 점을 생성 → 두 위성의 시점에서 픽셀 각을 계산(투영) → `tie_point_triangulate`로 재구성해 원래 좌표와 일치 확인.

In [ ]:
def project_to_spacecraft(point: np.ndarray, r_sc: np.ndarray, alpha_sep: float, which: str) -> tuple[float, float]:
    """Project a heliocentric 3D point onto the pixel angles a STEREO spacecraft would measure.

    The spacecraft's line-of-sight points from r_sc toward the Sun centre (origin). The
    X-direction angle alpha is measured in the XZ plane of the spacecraft frame, the
    Y-direction angle delta is measured perpendicular to that plane.

    Args:
        point: (3,) array giving the heliocentric (x, y, z) of the source, in km.
        r_sc: (3,) array giving the spacecraft heliocentric position, in km.
        alpha_sep: Spacecraft separation angle (positive for B, zero for A), radians.
        which: 'A' or 'B', controls which rotation is applied to transform into the
            spacecraft line-of-sight frame.

    Returns:
        (alpha, delta): Pixel angles in radians.
    """
    # Translate so the spacecraft is at the origin.
    v = point - r_sc
    # Rotate around Y-axis so the spacecraft's line-of-sight aligns with +Z.
    # For STEREO-A the line-of-sight is already along +Z; for STEREO-B, rotate by -alpha_sep.
    if which == "A":
        theta = 0.0
    elif which == "B":
        theta = -alpha_sep
    else:
        raise ValueError("which must be 'A' or 'B'")
    c, s = np.cos(theta), np.sin(theta)
    rot = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    v_sc = rot @ v
    # Pixel angles: alpha = X-offset angle, delta = Y-offset angle.
    alpha = np.arctan2(v_sc[0], v_sc[2])
    delta = np.arctan2(v_sc[1], v_sc[2])
    return alpha, delta


# Pick a 3D point (a coronal-loop apex at ~100 Mm above the surface, slightly off-centre).
target = np.array([5.0e5, 2.0e5, 0.0])  # km
print(f"Input  point: {target} km, altitude = {(np.linalg.norm(target) - R_SUN_KM)/1000:.1f} Mm")

alpha_sep_rad = np.deg2rad(30.0)
r_a, r_b = spacecraft_positions(D_A_KM, D_B_KM, 30.0)

a_a, d_a = project_to_spacecraft(target, r_a, alpha_sep_rad, "A")
a_b, d_b = project_to_spacecraft(target, r_b, alpha_sep_rad, "B")
print(f"  Seen from A: alpha = {np.rad2deg(a_a):+.5f} deg, delta = {np.rad2deg(d_a):+.5f} deg")
print(f"  Seen from B: alpha = {np.rad2deg(a_b):+.5f} deg, delta = {np.rad2deg(d_b):+.5f} deg")

x_rec, y_rec, z_rec, r_rec, h_rec = tie_point_triangulate(a_a, d_a, a_b, alpha_sep_rad)
print(f"\nRecovered point: ({x_rec:+.3e}, {y_rec:+.3e}, {z_rec:+.3e}) km")
print(f"Reconstruction error: {np.linalg.norm([x_rec, y_rec, z_rec] - target):.2e} km")

**English.** The reconstruction error should be well below the kilometre level, limited only by floating-point precision and (more importantly for real data) by pixel centroiding error. This demonstrates that the tie-point method is *exact* in the absence of noise.

**한국어.** 재구성 오차는 킬로미터 수준 이하로 나와야 하며, 부동소수점 정밀도만이 한계(실자료에서는 픽셀 중심측정 오차가 지배). 잡음이 없을 때 타이-포인트 법이 *정확*함을 보여준다.

## 5. Synthetic Coronal Loop: Full Triangulation Visualisation / 합성 코로나 루프 전체 시각화

**English.** Now we triangulate an entire coronal loop. The loop is modelled as a semi-circular arc in 3D (Aschwanden 2011 Eq. 25 in effect), projected onto both spacecraft views, and reconstructed point by point.

**한국어.** 이제 코로나 루프 전체를 삼각측량한다. 루프는 3D 반원 호(Aschwanden 2011 Eq. 25 상응), 양 위성 시점에 투영 후 점별 재구성.

In [ ]:
def synthetic_semicircular_loop(
    loop_half_length_km: float,
    footpoint_baseline_km: float,
    footpoint_xy: tuple[float, float],
    inclination_deg: float,
    n_points: int = 60,
) -> np.ndarray:
    """Generate a 3D semi-circular loop lying above the solar surface.

    The loop plane is tilted from the local vertical by inclination_deg. The loop apex
    follows h(s) = (2L/pi) sin(pi s / 2L) (Aschwanden 2011 Eq. 25).

    Args:
        loop_half_length_km: L, half length of the loop.
        footpoint_baseline_km: Distance between footpoints.
        footpoint_xy: (x0, y0) location of the baseline midpoint on the solar surface.
        inclination_deg: Inclination angle of the loop plane from local vertical.
        n_points: Number of sample points along the loop.

    Returns:
        Array (n_points, 3) of heliocentric (x, y, z) coordinates in km.
    """
    s = np.linspace(0, 2 * loop_half_length_km, n_points)
    # Height above surface.
    h = (2 * loop_half_length_km / np.pi) * np.sin(np.pi * s / (2 * loop_half_length_km))
    # Along-baseline coordinate.
    u = (s / (2 * loop_half_length_km) - 0.5) * footpoint_baseline_km
    # Apply inclination rotation within the local vertical plane.
    theta = np.deg2rad(inclination_deg)
    x_loop = u * np.cos(theta) - h * np.sin(theta)
    z_loop = u * np.sin(theta) + h * np.cos(theta)
    y_loop = np.zeros_like(s)
    # Place the baseline midpoint on the solar surface at (x0, y0, R_sun).
    x0, y0 = footpoint_xy
    points = np.column_stack([x0 + x_loop, y0 + y_loop, R_SUN_KM + z_loop])
    return points


loop_3d = synthetic_semicircular_loop(
    loop_half_length_km=150_000.0,  # L = 150 Mm -> full length 300 Mm.
    footpoint_baseline_km=120_000.0,
    footpoint_xy=(3.0e5, 1.5e5),
    inclination_deg=25.0,
    n_points=60,
)
print(f"Generated loop with {loop_3d.shape[0]} points")
print(f"Apex altitude: {(np.max(np.linalg.norm(loop_3d, axis=1)) - R_SUN_KM)/1000:.1f} Mm")

In [ ]:
# Project every loop point onto both STEREO-A and STEREO-B, then reconstruct.
alpha_sep_deg = 45.0
alpha_sep_rad = np.deg2rad(alpha_sep_deg)
r_a, r_b = spacecraft_positions(D_A_KM, D_B_KM, alpha_sep_deg)

a_a = np.array([project_to_spacecraft(p, r_a, alpha_sep_rad, "A")[0] for p in loop_3d])
d_a = np.array([project_to_spacecraft(p, r_a, alpha_sep_rad, "A")[1] for p in loop_3d])
a_b = np.array([project_to_spacecraft(p, r_b, alpha_sep_rad, "B")[0] for p in loop_3d])
d_b = np.array([project_to_spacecraft(p, r_b, alpha_sep_rad, "B")[1] for p in loop_3d])

loop_recon = np.array(
    [tie_point_triangulate(ai, di, bi, alpha_sep_rad)[:3] for ai, di, bi in zip(a_a, d_a, a_b)]
)

rms_error_km = np.sqrt(np.mean(np.sum((loop_recon - loop_3d) ** 2, axis=1)))
print(f"Separation angle used: {alpha_sep_deg} deg")
print(f"RMS reconstruction error: {rms_error_km:.3e} km  ({rms_error_km/1000:.3e} Mm)")

In [ ]:
# Visualise: two spacecraft views + reconstructed vs true 3D.
fig = plt.figure(figsize=(14, 5))

# Panel 1: STEREO-A projection (image plane).
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(np.rad2deg(a_a) * ARCSEC_PER_DEG, np.rad2deg(d_a) * ARCSEC_PER_DEG, "b-", lw=2)
ax1.set_xlabel(r"$\alpha$ [arcsec]")
ax1.set_ylabel(r"$\delta$ [arcsec]")
ax1.set_title("STEREO-A image / A 이미지")
ax1.set_aspect("equal")
ax1.grid(alpha=0.3)

# Panel 2: STEREO-B projection.
ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(np.rad2deg(a_b) * ARCSEC_PER_DEG, np.rad2deg(d_b) * ARCSEC_PER_DEG, "r-", lw=2)
ax2.set_xlabel(r"$\alpha$ [arcsec]")
ax2.set_ylabel(r"$\delta$ [arcsec]")
ax2.set_title("STEREO-B image / B 이미지")
ax2.set_aspect("equal")
ax2.grid(alpha=0.3)

# Panel 3: 3D reconstruction (subtracting R_sun for clearer axes).
ax3 = fig.add_subplot(1, 3, 3, projection="3d")
scale = 1e-3  # km -> Mm for plotting.
true_h = (np.linalg.norm(loop_3d, axis=1) - R_SUN_KM) * scale
ax3.plot(loop_3d[:, 0] * scale, loop_3d[:, 1] * scale, true_h, "k-", lw=3, label="true / 실제")
rec_h = (np.linalg.norm(loop_recon, axis=1) - R_SUN_KM) * scale
ax3.plot(loop_recon[:, 0] * scale, loop_recon[:, 1] * scale, rec_h, "g--", lw=1.5, label="triangulated / 삼각측량")
ax3.set_xlabel("x [Mm]")
ax3.set_ylabel("y [Mm]")
ax3.set_zlabel("h above surface [Mm]")
ax3.set_title("Tie-point reconstruction / 타이-포인트 재구성")
ax3.legend()

plt.tight_layout()
plt.show()

## 6. Sensitivity to Separation Angle / 분리각 민감도

**English.** The accuracy of triangulation depends on the stereoscopic separation angle. For alpha_sep near 0 or 180 degrees the two views degenerate — one expects large error. Let us inject Gaussian pixel noise and measure reconstruction RMS as a function of alpha_sep.

**한국어.** 삼각측량 정확도는 입체 분리각에 의존. alpha_sep이 0° 또는 180° 근처에서는 두 시점이 축퇴되어 오차가 커진다. 가우시안 픽셀 잡음을 주입하고 alpha_sep 함수로 재구성 RMS를 측정해 보자.

In [ ]:
def reconstruction_rms(alpha_sep_deg: float, sigma_arcsec: float, rng: np.random.Generator, n_trials: int = 100) -> float:
    """Return Monte Carlo mean RMS error for triangulating a single apex point.

    Args:
        alpha_sep_deg: Spacecraft separation angle in degrees.
        sigma_arcsec: Standard deviation of Gaussian pixel noise in arcsec.
        rng: Numpy random generator instance.
        n_trials: Number of noise realisations.

    Returns:
        RMS reconstruction error in km.
    """
    sigma_rad = np.deg2rad(sigma_arcsec / ARCSEC_PER_DEG)
    alpha_sep_rad = np.deg2rad(alpha_sep_deg)
    r_a, r_b = spacecraft_positions(D_A_KM, D_B_KM, alpha_sep_deg)

    target = np.array([5.0e5, 2.0e5, 0.0])
    a_a_true, d_a_true = project_to_spacecraft(target, r_a, alpha_sep_rad, "A")
    a_b_true, _ = project_to_spacecraft(target, r_b, alpha_sep_rad, "B")

    errors = []
    for _ in range(n_trials):
        a_a = a_a_true + rng.normal(0, sigma_rad)
        d_a = d_a_true + rng.normal(0, sigma_rad)
        a_b = a_b_true + rng.normal(0, sigma_rad)
        x, y, z, _, _ = tie_point_triangulate(a_a, d_a, a_b, alpha_sep_rad)
        errors.append(np.linalg.norm([x, y, z] - target))
    return float(np.sqrt(np.mean(np.array(errors) ** 2)))


rng = np.random.default_rng(seed=42)
sep_grid_deg = np.array([1, 2, 5, 7.3, 10, 20, 45, 90, 135, 170, 179])
# Typical EUVI centroiding error ~0.5 arcsec.
rms = np.array([reconstruction_rms(s, sigma_arcsec=0.5, rng=rng) for s in sep_grid_deg])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sep_grid_deg, rms / 1000, "o-", lw=2, ms=8)
ax.set_xlabel(r"Separation angle $\alpha_{\rm sep}$ [deg]")
ax.set_ylabel("Reconstruction RMS [Mm]")
ax.set_title("Sensitivity of triangulation to STEREO separation angle / 분리각에 대한 민감도")
ax.set_yscale("log")
ax.grid(alpha=0.3, which="both")
ax.axvline(7.3, color="r", ls="--", alpha=0.6, label="2007 May 9 (first STEREO triangulation)")
ax.legend()
plt.tight_layout()
plt.show()

for s, e in zip(sep_grid_deg, rms):
    print(f"  alpha_sep = {s:6.1f} deg  ->  RMS error = {e/1000:.2f} Mm")

**English.** Observation: reconstruction is best around separation angles of ~20-60 degrees; it degrades strongly at very small (≲3 deg) and very large (≳170 deg) separations. This matches Aschwanden's operational recommendation that large-angle stereoscopy was best in the middle years of the STEREO mission (2008-2010).

**한국어.** 관찰: 재구성은 분리각 ~20°–60° 부근에서 최적이며, 매우 작은(≲3°) 또는 매우 큰(≳170°) 분리에서 급격히 악화. 이는 Aschwanden이 권고한 STEREO 임무 중기(2008–2010)의 대각 입체법 최적성과 일치.

## 7. Back-Projection Tomography Demo / 역투영 토모그래피 데모

**English.** We now demonstrate the simplest tomographic inversion: back-projection. A 2D phantom (a stylised coronal cross-section with two dense blobs) is projected from many angles (the Radon transform), then the back-projection sum is computed. This is the scheme illustrated in Aschwanden (2011) Figure 4 / Davila & Thompson (1992).

**한국어.** 가장 단순한 토모그래피 반전인 역투영(back-projection)을 시연. 2D 팬텀(두 밀집 덩어리가 있는 양식화된 코로나 단면)을 여러 각도로 투영(Radon 변환)한 후 역투영 합을 계산. 이는 Aschwanden (2011) Figure 4 / Davila & Thompson (1992)의 방식.

In [ ]:
try:
    from skimage.transform import radon, iradon
    SKIMAGE_OK = True
except ImportError:
    SKIMAGE_OK = False
    print("scikit-image unavailable; using a hand-written projection loop.")


def make_coronal_phantom(n: int = 128) -> np.ndarray:
    """Create a simple 2D coronal-density phantom on an n x n grid.

    The phantom mimics a limb view of two dense coronal loop arcades at different heights.

    Args:
        n: Linear grid size.

    Returns:
        (n, n) float array with values in [0, 1].
    """
    xg, yg = np.meshgrid(np.linspace(-1, 1, n), np.linspace(-1, 1, n))
    phantom = np.zeros((n, n), dtype=float)
    # Two Gaussian density enhancements (representing over-dense active region cores).
    phantom += 0.9 * np.exp(-((xg - 0.25) ** 2 + (yg - 0.15) ** 2) / (2 * 0.08 ** 2))
    phantom += 0.6 * np.exp(-((xg + 0.30) ** 2 + (yg - 0.05) ** 2) / (2 * 0.12 ** 2))
    # A diffuse background, limb-brightened toward one side.
    phantom += 0.15 * np.exp(-(xg ** 2 + yg ** 2) / (2 * 0.45 ** 2))
    return phantom / phantom.max()


phantom = make_coronal_phantom(128)
print(f"Phantom shape: {phantom.shape}, max value: {phantom.max():.3f}")

In [ ]:
def simple_back_project(sinogram: np.ndarray, angles_deg: np.ndarray, output_size: int) -> np.ndarray:
    """Back-project a sinogram without any filtering (simplest tomographic inversion).

    For each angle, the projection is smeared back across the image at that same angle
    and the contributions are summed. No filter is applied, so the result is blurred by
    a 1/r point-spread function — this is the classical back-projection reconstruction
    illustrated in Aschwanden (2011) Figure 4.

    Args:
        sinogram: Shape (n_detectors, n_angles) sinogram.
        angles_deg: Projection angles in degrees.
        output_size: Size of the square output image.

    Returns:
        (output_size, output_size) back-projected image.
    """
    recon = np.zeros((output_size, output_size), dtype=float)
    xg, yg = np.meshgrid(
        np.linspace(-output_size / 2, output_size / 2 - 1, output_size),
        np.linspace(-output_size / 2, output_size / 2 - 1, output_size),
    )
    centre = sinogram.shape[0] // 2
    for i, theta in enumerate(angles_deg):
        t = np.deg2rad(theta)
        # Project each (x, y) to its detector coordinate.
        t_coord = xg * np.cos(t) + yg * np.sin(t) + centre
        t_idx = np.clip(np.round(t_coord).astype(int), 0, sinogram.shape[0] - 1)
        recon += sinogram[t_idx, i]
    return recon / len(angles_deg)


# Generate the sinogram (Radon transform).
n_angles_list = [4, 12, 45, 180]
fig, axes = plt.subplots(2, len(n_angles_list) + 1, figsize=(3.2 * (len(n_angles_list) + 1), 6))

# Column 0: true phantom.
axes[0, 0].imshow(phantom, cmap="inferno")
axes[0, 0].set_title("True phantom / 실제")
axes[0, 0].axis("off")
axes[1, 0].axis("off")
axes[1, 0].text(0.5, 0.5, "n_angles\n= ? (true)", ha="center", va="center", fontsize=12)

for col, n_angles in enumerate(n_angles_list, start=1):
    angles = np.linspace(0, 180, n_angles, endpoint=False)
    if SKIMAGE_OK:
        sino = radon(phantom, theta=angles, circle=True)
        recon_fb = iradon(sino, theta=angles, filter_name=None, circle=True)  # no filter
    else:
        from scipy.ndimage import rotate
        proj_list = [rotate(phantom, -a, reshape=False, order=1).sum(axis=0) for a in angles]
        sino = np.stack(proj_list, axis=1)
        recon_fb = simple_back_project(sino, angles, output_size=phantom.shape[0])
    axes[0, col].imshow(sino, cmap="gray", aspect="auto")
    axes[0, col].set_title(f"Sinogram ({n_angles} views) / 시노그램")
    axes[0, col].set_xlabel("angle index")
    axes[0, col].set_ylabel("detector")
    axes[1, col].imshow(recon_fb, cmap="inferno")
    axes[1, col].set_title(f"Back-projected {n_angles} / 역투영")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

**English.** Observations:
- With only 4 projections (the best STEREO-era case: A + B + SOHO + SDO), the reconstruction is a heavily blurred superposition — explicitly the under-constrained regime Aschwanden (2011) warns about in Section 3.5.
- With 45+ projections (achievable only via solar rotation under the assumption of stationarity), the back-projection converges on the original phantom — but only approximately, because this demonstration uses *unfiltered* back-projection. Real algorithms (regularized positive-estimation, Frazin 2000) perform much better.

**한국어.** 관찰:
- 4개 투영만으로는(STEREO 시대 최대: A + B + SOHO + SDO) 재구성이 심하게 흐려진 중첩 — Aschwanden (2011) Section 3.5가 경고하는 under-constrained 영역.
- 45개 이상 투영이면(정상성 가정 하 태양 자전 활용) 역투영이 원래 팬텀에 수렴 — 그러나 여기서는 *필터 없는* 역투영이라 근사적. 실제 알고리즘(regularized positive-estimation, Frazin 2000)은 훨씬 우수.

## 8. Summary / 요약

**English.** What we implemented:
1. **Spacecraft geometry** — STEREO-A and STEREO-B positions in a heliocentric frame (Section 2).
2. **Analytic tie-point triangulation** — the closed-form formulas of Aschwanden (2011) Eqs. 15-23 (Section 3), validated by a round-trip projection test.
3. **Full loop triangulation** with a synthetic semi-circular loop, showing how 2D projections from two viewpoints yield correct 3D reconstruction (Section 5).
4. **Sensitivity analysis** showing the reconstruction error as a function of separation angle — confirming the paper's operational guidance that 20-60 degrees is ideal (Section 6).
5. **Back-projection tomography demo** on a 2D coronal phantom, illustrating why under-constrained inversion (few STEREO projections) fails (Section 7).

**한국어.** 구현한 내용:
1. **위성 기하** — 태양 중심 좌표계에서 STEREO-A, STEREO-B 위치 (Section 2).
2. **해석적 타이-포인트 삼각측량** — Aschwanden (2011) Eqs. 15–23의 닫힌 꼴 공식 (Section 3), 왕복 투영 테스트로 검증.
3. **합성 반원 루프 전체의 삼각측량** — 두 시점의 2D 투영이 정확한 3D 재구성을 주는지 확인 (Section 5).
4. **분리각 민감도** — 분리각 함수로서 재구성 오차. 논문의 20°–60° 최적 권고와 일치 (Section 6).
5. **2D 코로나 팬텀의 역투영 토모그래피** — 왜 under-constrained 반전(몇 개의 STEREO 투영)이 실패하는지 설명 (Section 7).

### Next steps / 다음 단계

- Implement magnetic-field-constrained triangulation (Aschwanden 2011 Section 3.4) by extending the tie-point method with a potential-field prior.
- Implement the ISTAR forward-fitting pipeline (Aschwanden et al. 2009c) to recover a full 3D active-region DEM.
- Compare unfiltered back-projection with the filtered (Ram-Lak, Hann) variants and the positive-estimation method of Frazin (2000).

- potential-field prior를 추가한 자기장 제약 삼각측량 (Section 3.4) 구현.
- ISTAR forward-fitting 파이프라인(Aschwanden et al. 2009c)으로 활동영역 3D DEM 복원.
- 필터 없는 역투영 vs. 필터링(Ram-Lak, Hann) 변형 vs. Frazin(2000) positive-estimation 비교.